# Decode RSC head direction with a small PyTorch MLP and checkpoints

This companion notebook is deliberately small: one cylinder session, one cylinder window, one neural network. It mirrors the early data-prep logic in your MATLAB decoder, then adds disk checkpoints so the best model can be restored after a notebook restart.

The scientific question for this first pass is simple: given RSC calcium activity at one imaging frame, can a neural network predict the animal's head direction?

## What this notebook teaches

- Loading MATLAB/HDF5 data from Python
- Inspecting arrays with `numpy` and `pandas`
- Making `torch.Tensor` objects
- Building a `Dataset` and `DataLoader`
- Defining an `nn.Module`
- Training with `loss.backward()` and `optimizer.step()`
- Saving and loading PyTorch checkpoints / PyTorch checkpoints
- Evaluating circular decoding error
- Plotting actual versus decoded HD

In [ ]:
# If imports fail, run this in a terminal from the project folder:
# pip install -r requirements.txt

from pathlib import Path
import math
import random

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

plt.style.use("seaborn-v0_8-whitegrid")
torch.set_float32_matmul_precision("high")

def seed_everything(seed=7):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Choose one session and one cylinder window

The MATLAB script used `sessionList.xlsx` plus robust-cell tables to loop over many sessions. For learning PyTorch, start with one known session and all accepted CNMF-E components. Later, you can add robust-cell selection and cross-session loops.

In [ ]:
DATA_ROOT = Path(r"D:/Shiyun_RSC_1p")
SESSION_NAME = "STIM1_2026_01_22_constantCue_acquisition"
SESSION_DIR = DATA_ROOT / SESSION_NAME

SESSION_DATA = SESSION_DIR / "sessionData.mat"
CNMFE_HDF5 = SESSION_DIR / "pipeline_cnmfe_results_2026-01-24_08-18.hdf5"

# MATLAB window labels: 0=pre-rotation, 1=post-rotation, 2=dark in Python indexing.
WINDOW_INDEX = 0
WINDOW_NAMES = ["pre rotation", "post rotation", "dark"]

# Set to None for no speed filtering in the first learning pass.
# MATLAB used speed_cri.of after lag alignment; you can add that later.
MIN_SPEED_PX_PER_S = None

assert SESSION_DATA.exists(), SESSION_DATA
assert CNMFE_HDF5.exists(), CNMFE_HDF5
SESSION_DATA, CNMFE_HDF5

## MATLAB v7.3/HDF5 helpers

MATLAB v7.3 files are HDF5 files. Arrays are often stored with dimensions reversed relative to MATLAB display, so the helper below transposes 2D numeric datasets by default.

In [ ]:
def read_matlab_h5_array(group, key, transpose_2d=True):
    """Read a numeric dataset from a MATLAB v7.3 struct group."""
    arr = np.array(group[key])
    arr = np.squeeze(arr)
    if transpose_2d and arr.ndim == 2:
        arr = arr.T
    return arr


def describe_h5_group(group, max_items=30):
    rows = []
    for key in list(group.keys())[:max_items]:
        obj = group[key]
        shape = getattr(obj, "shape", "")
        dtype = getattr(obj, "dtype", "")
        rows.append((key, shape, dtype))
    return pd.DataFrame(rows, columns=["key", "shape", "dtype"])


with h5py.File(SESSION_DATA, "r") as f:
    print(list(f.keys()))
    display(describe_h5_group(f["sessionData"]))
    display(describe_h5_group(f["sessionData"]["of"]))

In [ ]:
def load_session_arrays(session_data_path, cnmfe_hdf5_path):
    with h5py.File(session_data_path, "r") as f:
        sd = f["sessionData"]
        of = sd["of"]

        calcium_time = read_matlab_h5_array(sd, "calcium_time", transpose_2d=False).astype(float)
        raw_C = read_matlab_h5_array(sd, "raw_C", transpose_2d=True).astype(np.float32)
        hd = read_matlab_h5_array(of, "hd", transpose_2d=False).astype(float)
        segment_time = read_matlab_h5_array(of, "segment_time", transpose_2d=True).astype(float)
        raw_coord = read_matlab_h5_array(of, "raw_coord", transpose_2d=True).astype(float)

    with h5py.File(cnmfe_hdf5_path, "r") as f:
        if "/estimates/idx_components" in f:
            accepted = np.array(f["/estimates/idx_components"]).astype(int).ravel()
        else:
            accepted = np.arange(raw_C.shape[1])

    # MATLAB added +1 because MATLAB is 1-based. Python keeps the HDF5 indices as 0-based.
    accepted = accepted[(accepted >= 0) & (accepted < raw_C.shape[1])]
    C = raw_C[:, accepted]
    C[C < 0] = 0

    return {
        "calcium_time": calcium_time,
        "C": C,
        "hd": hd,
        "segment_time": segment_time,
        "coord": raw_coord,
        "accepted_components": accepted,
    }


data = load_session_arrays(SESSION_DATA, CNMFE_HDF5)
{k: np.shape(v) for k, v in data.items() if k != "accepted_components"}, len(data["accepted_components"])

## Match the MATLAB open-field and cylinder-window preprocessing

In the MATLAB code, open field is defined from the first open-field segment start through the second-to-last segment end. Calcium is min-max normalized within that open-field epoch. Then one cylinder window is selected from `segment_time`.

In [ ]:
def minmax_normalize_columns(X):
    X = X.astype(np.float32, copy=True)
    lo = np.nanmin(X, axis=0, keepdims=True)
    hi = np.nanmax(X, axis=0, keepdims=True)
    denom = hi - lo
    denom[denom == 0] = 1
    return (X - lo) / denom


def circular_difference(a, b):
    return np.angle(np.exp(1j * (a - b)))


def compute_speed(coord, time_ms):
    dxy = np.diff(coord, axis=0)
    dt = np.diff(time_ms) / 1000.0
    speed = np.full(len(time_ms), np.nan)
    speed[1:] = np.sqrt((dxy ** 2).sum(axis=1)) / dt
    return speed


def prepare_window(data, window_index=0, min_speed_px_per_s=None):
    calcium_time = data["calcium_time"]
    segment_time = data["segment_time"]

    of_start = segment_time[0, 0]
    of_end = segment_time[-2, 1]
    of_flag = (calcium_time >= of_start) & (calcium_time <= of_end)

    of_time = calcium_time[of_flag]
    of_C = minmax_normalize_columns(data["C"][of_flag])
    of_hd = data["hd"][of_flag]
    of_coord = data["coord"][of_flag]

    win_start, win_end = segment_time[window_index]
    win_flag = (of_time >= win_start) & (of_time <= win_end)

    X = of_C[win_flag]
    hd = of_hd[win_flag]
    time = of_time[win_flag]
    coord = of_coord[win_flag]

    finite = np.isfinite(hd) & np.all(np.isfinite(X), axis=1)
    if min_speed_px_per_s is not None:
        speed = compute_speed(coord, time)
        finite &= speed >= min_speed_px_per_s

    return X[finite], hd[finite], time[finite], coord[finite]


X, hd, time_ms, coord = prepare_window(data, WINDOW_INDEX, MIN_SPEED_PX_PER_S)
print(WINDOW_NAMES[WINDOW_INDEX])
print(f"frames: {X.shape[0]:,}")
print(f"accepted cells: {X.shape[1]:,}")
print(f"duration: {(time_ms[-1] - time_ms[0]) / 1000 / 60:.2f} min")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
axes[0].hist(np.rad2deg(hd), bins=72, color="0.25")
axes[0].set_xlabel("HD (deg)")
axes[0].set_ylabel("frames")
axes[0].set_title("HD sampling")

sc = axes[1].scatter(coord[:, 0], coord[:, 1], s=4, c=np.rad2deg(hd), cmap="twilight", alpha=0.8)
axes[1].set_aspect("equal", adjustable="box")
axes[1].set_title("Cylinder path colored by HD")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")

cbar = fig.colorbar(sc,ax=axes[1])
cbar.set_label("HD (deg)")

plt.show()

## Train/validation/test split

The MATLAB decoder used contiguous cross-validation folds. Here we use contiguous temporal blocks too, because random frame splits can leak slow calcium dynamics from neighboring frames into both train and test.

In [ ]:
def temporal_split(n, train_frac=0.70, val_frac=0.15):
    train_end = int(n * train_frac)
    val_end = int(n * (train_frac + val_frac))
    return np.arange(0, train_end), np.arange(train_end, val_end), np.arange(val_end, n)


train_idx, val_idx, test_idx = temporal_split(len(hd))

#compute mu and sigma using only the training set
mu = X[train_idx].mean(axis=0, keepdims=True)
sigma = X[train_idx].std(axis=0, keepdims=True)
sigma[sigma == 0] = 1
Xz = (X - mu) / sigma #Then apply them to train, validation, and test

y = np.column_stack([np.cos(hd), np.sin(hd)]).astype(np.float32)

print(len(train_idx), len(val_idx), len(test_idx))
print(Xz.shape, y.shape)

In [ ]:
class FrameHDDataset(Dataset):
    def __init__(self, X, y, indices):
        self.X = torch.as_tensor(X[indices], dtype=torch.float32)
        self.y = torch.as_tensor(y[indices], dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


batch_size = 256
train_loader = DataLoader(FrameHDDataset(Xz, y, train_idx), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(FrameHDDataset(Xz, y, val_idx), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(FrameHDDataset(Xz, y, test_idx), batch_size=batch_size, shuffle=False)

## Define the MLP decoder

The model predicts a 2D vector. During evaluation, we convert it back to an angle with `atan2(pred_sin, pred_cos)`. Normalizing the output vector prevents large vector length from mattering more than angle.

In [ ]:
model = HDDecoderMLP(X.shape[1]).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(exist_ok=True)
checkpoint_path = checkpoint_dir / f"{SESSION_NAME}_{WINDOW_NAMES[WINDOW_INDEX].replace(' ', '_')}_mlp_best.pt"

model

In [ ]:
def run_epoch(model, loader, train=False):
    model.train(train) #turns dropout on/off
    total_loss = 0.0
    n = 0

    for xb, yb in loader: #loop through batches
        xb = xb.to(device)
        yb = yb.to(device)

        with torch.set_grad_enabled(train):
            pred = model(xb)
            loss = loss_fn(pred, yb)

        if train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * len(xb)
        n += len(xb)

    return total_loss / n


history = []
best_val = float("inf")
best_state = None
best_epoch = None
patience = 20
bad_epochs = 0

for epoch in range(1, 201):
    train_loss = run_epoch(model, train_loader, train=True)
    val_loss = run_epoch(model, val_loader, train=False)
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})

    if val_loss < best_val:
        best_val = val_loss
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad_epochs = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val": best_val,
                "history": history,
                "session_name": SESSION_NAME,
                "window_index": WINDOW_INDEX,
                "window_name": WINDOW_NAMES[WINDOW_INDEX],
                "n_cells": X.shape[1],
                "batch_size": batch_size,
                "model_class": "HDDecoderMLP",
                "optimizer_class": "AdamW",
                "learning_rate": optimizer.param_groups[0]["lr"],
                "weight_decay": optimizer.param_groups[0]["weight_decay"],
            },
            checkpoint_path,
        )
    else:
        bad_epochs += 1

    if epoch == 1 or epoch % 10 == 0:
        print(f"epoch {epoch:03d} train={train_loss:.4f} val={val_loss:.4f}")

    if bad_epochs >= patience:
        print(f"early stopping at epoch {epoch}")
        break

model.load_state_dict(best_state)
print(f"best epoch: {best_epoch}, best validation MSE: {best_val:.4f}")
print(f"saved checkpoint: {checkpoint_path}")
hist = pd.DataFrame(history)
hist.tail()

## Load the best checkpoint / 加载最佳检查点

This cell shows the disk-based version of `best_state`. You can restart the kernel, recreate the model architecture, and then load the saved checkpoint / 检查点 from disk.


In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

loaded_model = HDDecoderMLP(checkpoint["n_cells"]).to(device)
loaded_model.load_state_dict(checkpoint["model_state_dict"])
loaded_model.eval()

loaded_optimizer = torch.optim.AdamW(
    loaded_model.parameters(),
    lr=checkpoint["learning_rate"],
    weight_decay=checkpoint["weight_decay"],
)
loaded_optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

print(f"loaded epoch: {checkpoint['epoch']}")
print(f"loaded best validation MSE: {checkpoint['best_val']:.4f}")
print(f"loaded session: {checkpoint['session_name']} - {checkpoint['window_name']}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(hist["epoch"], hist["train_loss"], label="train")
ax.plot(hist["epoch"], hist["val_loss"], label="validation")
ax.set_xlabel("epoch")
ax.set_ylabel("MSE on [cos, sin]")
ax.legend()
ax.set_title("Training curve")
plt.show()

## Decode HD and score circular error

This follows the same circular-error idea as the MATLAB code: decoded minus actual is wrapped onto `[-pi, pi]`, then summarized in degrees.

In [ ]:
@torch.no_grad() #turns off gradint tracking during prediction
def predict_vectors(model, X_array, batch_size=1024):
    model.eval()
    preds = []
    for start in range(0, len(X_array), batch_size):
        xb = torch.as_tensor(X_array[start:start + batch_size], dtype=torch.float32, device=device)
        preds.append(model(xb).cpu().numpy())
    return np.vstack(preds)


pred_vec = predict_vectors(loaded_model, Xz[test_idx])
pred_hd = np.arctan2(pred_vec[:, 1], pred_vec[:, 0])
actual_hd = hd[test_idx]
err = circular_difference(pred_hd, actual_hd)
abs_err_deg = np.abs(np.rad2deg(err))

print(f"mean absolute circular error: {abs_err_deg.mean():.1f} deg")
print(f"median absolute circular error: {np.median(abs_err_deg):.1f} deg")

In [ ]:
t_s = (time_ms[test_idx] - time_ms[test_idx][0]) / 1000

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True, constrained_layout=True)
axes[0].plot(t_s, np.rad2deg(actual_hd), lw=1, color="black")
axes[0].set_ylabel("actual HD (deg)")
axes[0].set_ylim(-190, 190)

axes[1].plot(t_s, np.rad2deg(pred_hd), lw=1, color="tab:blue")
axes[1].set_ylabel("decoded HD (deg)")
axes[1].set_ylim(-190, 190)

axes[2].plot(t_s, np.rad2deg(err), lw=1, color="tab:red")
axes[2].axhline(0, color="0.2", lw=1)
axes[2].set_ylabel("error (deg)")
axes[2].set_xlabel("test time (s)")
axes[2].set_ylim(-190, 190)

fig.suptitle(f"{SESSION_NAME} - {WINDOW_NAMES[WINDOW_INDEX]}")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

axes[0].scatter(np.rad2deg(actual_hd), np.rad2deg(pred_hd), s=6, alpha=0.35)
axes[0].set_xlim(-180, 180)
axes[0].set_ylim(-180, 180)
axes[0].set_xlabel("actual HD (deg)")
axes[0].set_ylabel("decoded HD (deg)")
axes[0].set_title("Actual vs decoded")

axes[1].hist(abs_err_deg, bins=np.arange(0, 185, 5), color="tab:red", alpha=0.85)
axes[1].set_xlabel("absolute circular error (deg)")
axes[1].set_ylabel("test frames")
axes[1].set_title("Decoder error")

plt.show()

## Next extensions

Good next steps, in order:

1. Add the MATLAB lag-alignment step with circular interpolation.
2. Restrict input cells to the robust HD cells from `hold_robustCells.mat`.
3. Replace the single temporal split with 5 contiguous folds, matching `make_cv_partitions`.
4. Add a Gaussian Naive Bayes baseline in Python so MLP and Bayesian decoding are compared on exactly the same frames.
5. Try a temporal model, such as a 1D CNN over recent calcium history, after the frame-wise MLP is working.